In [ ]:
import os
subdir = ""
project_root = "/home/krzysztof/studia/magisterka/time-series-invariance"
os.chdir(os.path.join(project_root, subdir))

In [ ]:
import numpy as np


def convert(o):
    if isinstance(o, np.float32):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    raise TypeError(f"Type {type(o)} not serializable")



In [ ]:
import numpy as np

from src.embeddings.magnitude_spectrum import (
    band_energy_ratios,
    magnitude_spectrum,
    mfcc_features
)
from src.embeddings.seq2img_embeddings import s2i_vector
from src.embeddings.shape_based import paa, sax
from src.embeddings.simple_statistics import simple_statistics_vector
from src.embeddings.moments_1d import compute_1d_moment_vector

from notebooks.evaluations.helpers import (
    apply_embedding_function_shrunk,
    test_embeddings_quality,
    apply_embedding_function_shifts,

    )

import pandas as pd
import json

In [ ]:
def load_dataset(dataset_name):
    data_path = f"data/datasets/{dataset_name}.npz"
    data = np.load(data_path)
    return data["X"]

def load_distortions(dataset_name, distortion_type):
    data_path = f"data/distortions/{distortion_type}/{dataset_name}.npy"
    data = np.load(data_path)
    return data

In [ ]:
data_dir = "data/datasets"
dataset_names = [name[:-4] for name in os.listdir(data_dir)]
dataset_names

In [ ]:
def get_results(method_name, dataset_name, all_results, distortion_type, method):
    if method_name not in all_results:
        all_results[method_name] = {}

    data = load_dataset(dataset_name)
    random_shrunks = load_distortions(dataset_name, distortion_type)
    if distortion_type == "shift":
        embeddings = apply_embedding_function_shifts(data, method, random_shrunks)
    elif distortion_type == "shrink":
        embeddings = apply_embedding_function_shrunk(data, method, random_shrunks)
    else:
        raise ValueError(f"Unknown distortion type: {distortion_type}")
    print("embeddings shape:", embeddings.shape)

    _, results = test_embeddings_quality(embeddings)
    all_results[method_name][dataset_name] = results

In [ ]:
import pandas as pd

def get_result_df(all_results, metric_name):
    data_for_df = {}

    for method, datasets in all_results.items():
        row = {}
        for dataset, metrics in datasets.items():
            row[dataset] = metrics[metric_name]
        data_for_df[method] = row

    df = pd.DataFrame.from_dict(data_for_df, orient='index')

    return df

In [ ]:
def random_embedding(x):
    return np.random.rand(128)

## Shrunk invariance

In [ ]:
all_results_shrink_path = "results/shrink_results.json"
if os.path.exists(all_results_shrink_path):
    try:
        with open(all_results_shrink_path, "r") as f:
            all_results_shrink = json.load(f)
    except json.JSONDecodeError:
        print(f"Error decoding JSON from {all_results_shrink_path}. Starting with an empty dictionary.")
        all_results_shrink = {}
else:
    all_results_shrink = {}

### 1d moments

In [ ]:
method_name = "1d_moments"
for dataset_name in dataset_names:
    get_results(method_name, dataset_name, all_results_shrink, "shrink", compute_1d_moment_vector)
with open(all_results_shrink_path, "w") as f:
    json.dump(all_results_shrink, f, indent=4)

### Random embedding

In [ ]:
method_name = "random"
for dataset_name in dataset_names:
    get_results(method_name, dataset_name, all_results_shrink, "shrink", random_embedding)
with open(all_results_shrink_path, "w") as f:
    json.dump(all_results_shrink, f, indent=4)

### Simple embedding vector

In [ ]:
method_name = "simple_embedding_vector"
for dataset_name in dataset_names:
    get_results(method_name, dataset_name, all_results_shrink, "shrink", simple_statistics_vector)
with open(all_results_shrink_path, "w") as f:
    json.dump(all_results_shrink, f, indent=4)

### Magnitude spectrum
Because it works on fix bands it does terribly after shrinking.

In [ ]:
method_name = "band_energy_ratios"
bands_i = np.linspace(0, 1, 10)
bands = [(bands_i[i], bands_i[i+1]) for i in range(len(bands_i)-1)]

for dataset_name in dataset_names:
    get_results(method_name, dataset_name, all_results_shrink, "shrink", lambda x : band_energy_ratios(x, 1, bands))
with open(all_results_shrink_path, "w") as f:
    json.dump(all_results_shrink, f, indent=4)

In [ ]:
method_name = "mfcc_features"

for dataset_name in dataset_names:
    get_results(method_name, dataset_name, all_results_shrink, "shrink", lambda x : mfcc_features(x, 1, 12))
with open(all_results_shrink_path, "w") as f:
    json.dump(all_results_shrink, f, indent=4)

### Sequence to image hu vector

In [ ]:
img_name = "PSI"
method_name = f"seq2img_{img_name}"

for dataset_name in dataset_names:
    get_results(method_name, dataset_name, all_results_shrink, "shrink", lambda x: s2i_vector(x, name=img_name, psi_parameters=[(16, 4, 16)])[0])
with open(all_results_shrink_path, "w") as f:
    json.dump(all_results_shrink, f, indent=4)

In [ ]:
img_name = "MTF"
method_name = f"seq2img_{img_name}"

for dataset_name in dataset_names:
    get_results(method_name, dataset_name, all_results_shrink, "shrink", lambda x: s2i_vector(x, name=img_name, psi_parameters=[(16, 4, 16)])[0])
with open(all_results_shrink_path, "w") as f:
    json.dump(all_results_shrink, f, indent=4)

In [ ]:
img_name = "RP"
method_name = f"seq2img_{img_name}"

for dataset_name in dataset_names:
    get_results(method_name, dataset_name, all_results_shrink, "shrink", lambda x: s2i_vector(x, name=img_name, psi_parameters=[(16, 4, 16)])[0])
with open(all_results_shrink_path, "w") as f:
    json.dump(all_results_shrink, f, indent=4)

In [ ]:
img_name = "GAF"
method_name = f"seq2img_{img_name}"

for dataset_name in dataset_names:
    get_results(method_name, dataset_name, all_results_shrink, "shrink", lambda x: s2i_vector(x, name=img_name, psi_parameters=[(16, 4, 16)])[0])
with open(all_results_shrink_path, "w") as f:
    json.dump(all_results_shrink, f, indent=4)

In [ ]:
img_name = "STFT"
method_name = f"seq2img_{img_name}"

for dataset_name in dataset_names:
    get_results(method_name, dataset_name, all_results_shrink, "shrink", lambda x: s2i_vector(x, name=img_name, psi_parameters=[(16, 4, 16)], stft_parameters=[(64, 32)],)[0])
with open(all_results_shrink_path, "w") as f:
    json.dump(all_results_shrink, f, indent=4)

In [ ]:
img_name = "Plain_Tile"
method_name = f"seq2img_{img_name}"

for dataset_name in dataset_names:
    get_results(method_name, dataset_name, all_results_shrink, "shrink", lambda x: s2i_vector(x, name=img_name, psi_parameters=[(16, 4, 16)])[0])
with open(all_results_shrink_path, "w") as f:
    json.dump(all_results_shrink, f, indent=4)

### PAA

In [ ]:
method_name = "PAA"

for dataset_name in dataset_names:
    get_results(method_name, dataset_name, all_results_shrink, "shrink", lambda x: paa(x, 20))
with open(all_results_shrink_path, "w") as f:
    json.dump(all_results_shrink, f, indent=4)

### SAX

In [ ]:
method_name = "SAX"

for dataset_name in dataset_names:
    get_results(method_name, dataset_name, all_results_shrink, "shrink", lambda x: sax(x, 20, 5))
with open(all_results_shrink_path, "w") as f:
    json.dump(all_results_shrink, f, indent=4)

### ts2vec transfer

In [ ]:
all_results_shrink["ts2vec"] = {}
for dataset_name in dataset_names:
    embeddings = np.load(f"notebooks/evaluations/embeddings/ts2vec/shrink/{dataset_name}.npy")
    _, results = test_embeddings_quality(embeddings)
    for key in results:
        if isinstance(results[key], np.float32):
            results[key] = float(results[key])
    all_results_shrink["ts2vec"][dataset_name] = results
with open(all_results_shrink_path, "w") as f:
    json.dump(all_results_shrink, f, indent=4, default=convert)

In [ ]:
test_embeddings_quality(embeddings)

### TS-TCC

In [ ]:
all_results_shrink["ts-tcc-fine-tune"] = {}
for dataset_name in dataset_names:
    embeddings = np.load(f"notebooks/evaluations/embeddings/ts-tcc-fine-tune/shrink/{dataset_name}.npy")
    _, results = test_embeddings_quality(embeddings)
    for key in results:
        if isinstance(results[key], np.float32):
            results[key] = float(results[key])
    all_results_shrink["ts-tcc-fine-tune"][dataset_name] = results
with open(all_results_shrink_path, "w") as f:
    json.dump(all_results_shrink, f, indent=4, default=convert) 

In [ ]:
all_results_shrink["ts-tcc-self-supervised"] = {}
for dataset_name in dataset_names:
    embeddings = np.load(f"notebooks/evaluations/embeddings/ts-tcc-self-supervised/shrink/{dataset_name}.npy")
    _, results = test_embeddings_quality(embeddings)
    for key in results:
        if isinstance(results[key], np.float32):
            results[key] = float(results[key])
    all_results_shrink["ts-tcc-self-supervised"][dataset_name] = results
with open(all_results_shrink_path, "w") as f:
    json.dump(all_results_shrink, f, indent=4, default=convert) 

### Summary

In [ ]:
global_cos_sim_df = get_result_df(all_results_shrink, "global_cos_sim")
local_cos_sim_df = get_result_df(all_results_shrink, "local_cos_sim")
global_cos_sim_df

In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import itertools

fig, axes = plt.subplots(5, 4, figsize=(15, 15))
axes = axes.flatten()

num_datasets = len(dataset_names)

# Marker styles to cycle through
markers = ['o', 's', '^', 'D', 'v', 'P', 'X', '*', '<', '>', 'h', '8']
marker_cycle = itertools.cycle(markers)

# Assign a marker to each method
method_markers = {method: next(marker_cycle) for method in global_cos_sim_df.index}

for i, dataset_name in enumerate(dataset_names):
    for method_name in global_cos_sim_df.index:
        x = global_cos_sim_df.loc[method_name][dataset_name] * -1
        y = local_cos_sim_df.loc[method_name][dataset_name]

        x_jitter = x + np.random.normal(0, 0.0001, size=np.shape(x))
        y_jitter = y + np.random.normal(0, 0.0001, size=np.shape(y))

        axes[i].scatter(
            x_jitter,
            y_jitter,
            label=method_name,
            alpha=0.7,
            marker=method_markers[method_name]   # <- marker added
        )

    axes[i].set_xlabel("Global Cosine Similarity")
    axes[i].set_ylabel("Local Cosine Similarity")
    axes[i].set_title(f"Global vs Local Cosine Similarity for {dataset_name}")

# Hide unused axes
for j in range(num_datasets, len(axes)):
    axes[j].axis('off')

# Single legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower right', bbox_to_anchor=(0.95, 0.05))

plt.tight_layout()
plt.show()

In [ ]:
silhouette_score = get_result_df(all_results_shrink, "slihouette_score_max")
silhouette_score.style.highlight_max(axis=0)

In [ ]:
cohen_d = get_result_df(all_results_shrink, "cohen_d")
cohen_d.style.highlight_max(axis=0)

In [ ]:
fdr = get_result_df(all_results_shrink, "FDR")
fdr.style.highlight_max(axis=0)

## Shift invariance

In [ ]:
all_results_shift_path = "results/shift_results.json"
if os.path.exists(all_results_shift_path):
    with open(all_results_shift_path, "r") as f:
        all_results_shift = json.load(f)
else:
    all_results_shift = {}

### Random embedding

In [ ]:
method_name = "random"
for dataset_name in dataset_names:
    get_results(method_name, dataset_name, all_results_shift, "shift", random_embedding)
with open(all_results_shift_path, "w") as f:
    json.dump(all_results_shift, f, indent=4, default=convert)

### 1d moments

In [ ]:
method_name = "1d_moments"
for dataset_name in dataset_names:
    get_results(method_name, dataset_name, all_results_shift, "shift", compute_1d_moment_vector)
with open(all_results_shift_path, "w") as f:
    json.dump(all_results_shift, f, indent=4)

### Simple embedding vector

In [ ]:
method_name = "simple_embedding_vector"
for dataset_name in dataset_names:
    get_results(method_name, dataset_name, all_results_shift, "shift", simple_statistics_vector)
with open(all_results_shift_path, "w") as f:
    json.dump(all_results_shift, f, indent=4)

### Magnitude spectrum

In [ ]:
method_name = "band_energy_ratios"
bands_i = np.linspace(0, 1, 10)
bands = [(bands_i[i], bands_i[i+1]) for i in range(len(bands_i)-1)]

for dataset_name in dataset_names:
    get_results(method_name, dataset_name, all_results_shift, "shift", lambda x : band_energy_ratios(x, 1, bands))
with open(all_results_shift_path, "w") as f:
    json.dump(all_results_shift, f, indent=4)

In [ ]:
method_name = "mfcc_features"

for dataset_name in dataset_names:
    get_results(method_name, dataset_name, all_results_shift, "shift", lambda x : mfcc_features(x, 1, 12))
with open(all_results_shift_path, "w") as f:
    json.dump(all_results_shift, f, indent=4)

### Sequence to image hu vector

In [ ]:
img_name = "PSI"
method_name = f"seq2img_{img_name}"

for dataset_name in dataset_names:
    get_results(method_name, dataset_name, all_results_shift, "shift", lambda x: s2i_vector(x, name=img_name, psi_parameters=[(10, 4, 16)])[0])
with open(all_results_shift_path, "w") as f:
    json.dump(all_results_shift, f, indent=4)

In [ ]:
img_name = "MTF"
method_name = f"seq2img_{img_name}"

for dataset_name in dataset_names:
    get_results(method_name, dataset_name, all_results_shift, "shift", lambda x: s2i_vector(x, name=img_name, psi_parameters=[(16, 4, 16)], markov_bins=4)[0])
with open(all_results_shift_path, "w") as f:
    json.dump(all_results_shift, f, indent=4)

In [ ]:
img_name = "RP"
method_name = f"seq2img_{img_name}"

for dataset_name in dataset_names:
    get_results(method_name, dataset_name, all_results_shift, "shift", lambda x: s2i_vector(x, name=img_name, psi_parameters=[(16, 4, 16)])[0])
with open(all_results_shift_path, "w") as f:
    json.dump(all_results_shift, f, indent=4)

In [ ]:
img_name = "GAF"
method_name = f"seq2img_{img_name}"

for dataset_name in dataset_names:
    get_results(method_name, dataset_name, all_results_shift, "shift", lambda x: s2i_vector(x, name=img_name, psi_parameters=[(16, 4, 16)])[0])
with open(all_results_shift_path, "w") as f:
    json.dump(all_results_shift, f, indent=4)

In [ ]:
img_name = "STFT"
method_name = f"seq2img_{img_name}"

for dataset_name in dataset_names:
    get_results(method_name, dataset_name, all_results_shift, "shift", lambda x: s2i_vector(x, name=img_name, stft_parameters = [(16, 4)])[0])
with open(all_results_shift_path, "w") as f:
    json.dump(all_results_shift, f, indent=4)

In [ ]:
img_name = "Plain_Tile"
method_name = f"seq2img_{img_name}"

for dataset_name in dataset_names:
    get_results(method_name, dataset_name, all_results_shrink, "shrink", lambda x: s2i_vector(x, name=img_name, psi_parameters=[(16, 4, 16)])[0])
with open(all_results_shrink_path, "w") as f:
    json.dump(all_results_shrink, f, indent=4)

### PAA

In [ ]:
method_name = "PAA"

for dataset_name in dataset_names:
    get_results(method_name, dataset_name, all_results_shift, "shift", lambda x: paa(x, 20))
with open(all_results_shift_path, "w") as f:
    json.dump(all_results_shift, f, indent=4)

### SAX

In [ ]:
method_name = "SAX"

for dataset_name in dataset_names:
    get_results(method_name, dataset_name, all_results_shift, "shift", lambda x: sax(x, 20, 5))
with open(all_results_shift_path, "w") as f:
    json.dump(all_results_shift, f, indent=4)

### ts2vec transfer

In [ ]:
all_results_shift["ts2vec"] = {}
for dataset_name in dataset_names:
    embeddings = np.load(f"notebooks/evaluations/embeddings/ts2vec/shift/{dataset_name}.npy")
    _, results = test_embeddings_quality(embeddings)
    all_results_shift["ts2vec"][dataset_name] = results
with open(all_results_shift_path, "w") as f:
    json.dump(all_results_shift, f, indent=4, default=convert)

### TS-TCC

In [ ]:
all_results_shift["ts-tcc-fine-tune"] = {}
for dataset_name in dataset_names:
    embeddings = np.load(f"notebooks/evaluations/embeddings/ts-tcc-fine-tune/shift/{dataset_name}.npy")
    _, results = test_embeddings_quality(embeddings)
    all_results_shift["ts-tcc-fine-tune"][dataset_name] = results
with open(all_results_shift_path, "w") as f:
    json.dump(all_results_shift, f, indent=4, default=convert)

In [ ]:
all_results_shift["ts-tcc-self-supervised"] = {}
for dataset_name in dataset_names:
    embeddings = np.load(f"notebooks/evaluations/embeddings/ts-tcc-self-supervised/shift/{dataset_name}.npy")
    _, results = test_embeddings_quality(embeddings)
    all_results_shift["ts-tcc-self-supervised"][dataset_name] = results
with open(all_results_shift_path, "w") as f:
    json.dump(all_results_shift, f, indent=4, default=convert)

### Summary

In [ ]:
global_cos_sim_df = get_result_df(all_results_shift, "global_cos_sim")
local_cos_sim_df = get_result_df(all_results_shift, "local_cos_sim")
global_cos_sim_df

In [ ]:
from matplotlib import pyplot as plt

fig, axes = plt.subplots(5, 4, figsize=(15, 15))
axes = axes.flatten()

num_datasets = len(dataset_names)

for i, dataset_name in enumerate(dataset_names):
    for method_name in global_cos_sim_df.index:
        x = global_cos_sim_df.loc[method_name][dataset_name] * -1
        y = local_cos_sim_df.loc[method_name][dataset_name]
        x_jitter = x + np.random.normal(0, 0.0001, size=np.shape(x))
        y_jitter = y + np.random.normal(0, 0.0001, size=np.shape(y))
        axes[i].scatter(
            x_jitter,
            y_jitter,
            label=method_name,
            alpha=0.7
        )
    axes[i].set_xlabel("Global Cosine Similarity")
    axes[i].set_ylabel("Local Cosine Similarity")
    axes[i].set_title(f"Global vs Local Cosine Similarity for {dataset_name}")

# Hide any unused axes
for j in range(num_datasets, len(axes)):
    axes[j].axis('off')  # or axes[j].set_visible(False)

# Single legend at bottom right
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower right', bbox_to_anchor=(0.95, 0.05))

plt.tight_layout()
plt.show()

In [ ]:
cohen_d = get_result_df(all_results_shift, "cohen_d")
cohen_d.style.highlight_max(axis=0)

In [ ]:
fdr = get_result_df(all_results_shift, "FDR")
fdr.style.highlight_max(axis=0)

## Shift-shrunk tradeoff

In [ ]:
fdr_shift = get_result_df(all_results_shift, "FDR")
fdr_shrink = get_result_df(all_results_shrink, "FDR")

In [ ]:
from matplotlib import pyplot as plt
import itertools

fig, axes = plt.subplots(5, 4, figsize=(15, 15))
axes = axes.flatten()

num_datasets = len(dataset_names)

markers = ['o', 's', '^', 'D', 'v', 'P', 'X', '*', '<', '>', 'h', '8']
marker_cycle = itertools.cycle(markers)

# Assign a marker to each method
method_markers = {method: next(marker_cycle) for method in fdr_shift.index}

for i, dataset_name in enumerate(dataset_names):
    for method_name in fdr_shift.index:
        x = fdr_shift.loc[method_name][dataset_name]
        y = fdr_shrink.loc[method_name][dataset_name]
        x_jitter = x + np.random.normal(0, 0.0001, size=np.shape(x))
        y_jitter = y + np.random.normal(0, 0.0001, size=np.shape(y))
        axes[i].scatter(
            x_jitter,
            y_jitter,
            label=method_name,
            alpha=0.7,
            marker=method_markers[method_name]   # <- marker added
        )
    axes[i].set_xlabel("FDR for shifted data")
    axes[i].set_ylabel("FDR for shrinked data")
    axes[i].set_title(f"{dataset_name}")

# Hide any unused axes
for j in range(num_datasets, len(axes)):
    axes[j].axis('off')  # or axes[j].set_visible(False)

# Single legend at bottom right
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels,
           loc='lower right',
           bbox_to_anchor=(0.95, 0.05),
           ncol=2)
fig.suptitle("FDR for shifted vs shrinked data across datasets", y=0.98, fontsize=20)

plt.tight_layout(rect=[0, 0, 1, 0.96])  # leaves space at the top
plt.savefig("figures/fdr_shift_vs_shrink.png")
plt.show()

In [ ]:
from matplotlib import pyplot as plt
import itertools
import numpy as np
from matplotlib.lines import Line2D

fig, axes = plt.subplots(5, 4, figsize=(15, 15))
axes = axes.flatten()

num_datasets = len(dataset_names)

markers = ['o', 's', '^', 'D', 'v', 'P', 'X', '*', '<', '>', 'h', '8']
marker_cycle = itertools.cycle(markers)

# Assign a marker to each method
method_markers = {method: next(marker_cycle) for method in fdr_shift.index}

# --- Pareto function (robust, maximization) ---
def pareto_front_max(xs, ys):
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    n = len(xs)
    is_pareto = np.ones(n, dtype=bool)

    for i in range(n):
        for j in range(n):
            if i != j:
                if (xs[j] >= xs[i] and ys[j] >= ys[i]) and (xs[j] > xs[i] or ys[j] > ys[i]):
                    is_pareto[i] = False
                    break
    return is_pareto

for i, dataset_name in enumerate(dataset_names):

    # --- collect points FIRST (no change to your logic, just added) ---
    xs = []
    ys = []
    for method_name in fdr_shift.index:
        xs.append(fdr_shift.loc[method_name][dataset_name])
        ys.append(fdr_shrink.loc[method_name][dataset_name])

    xs = np.array(xs, dtype=float)
    ys = np.array(ys, dtype=float)

    pareto_mask = pareto_front_max(xs, ys)

    # --- your original loop (unchanged except index tracking) ---
    for j, method_name in enumerate(fdr_shift.index):
        x = xs[j]
        y = ys[j]

        x_jitter = x + np.random.normal(0, 0.0001, size=np.shape(x))
        y_jitter = y + np.random.normal(0, 0.0001, size=np.shape(y))

        axes[i].scatter(
            x_jitter,
            y_jitter,
            label=method_name,
            alpha=0.7,
            marker=method_markers[method_name]
        )

    # --- ADD: Pareto front overlay ---
    axes[i].scatter(
        xs[pareto_mask],
        ys[pareto_mask],
        s=140,
        facecolors='none',
        edgecolors='grey',
        linewidths=2,
        zorder=5
    )

    axes[i].set_xlabel("FDR for shifted data")
    axes[i].set_ylabel("FDR for shrinked data")
    axes[i].set_title(f"{dataset_name}")

# Hide any unused axes
for j in range(num_datasets, len(axes)):
    axes[j].axis('off')

# Single legend at bottom right
handles, labels = axes[0].get_legend_handles_labels()

pareto_handle = Line2D(
    [0], [0],
    marker='o',
    color='grey',
    markerfacecolor='none',
    linestyle='None',
    markersize=10,
    markeredgewidth=2,
    label='Pareto front'
)

handles.append(pareto_handle)
labels.append('Pareto front')

fig.legend(handles, labels,
           loc='lower right',
           bbox_to_anchor=(0.95, 0.05),
           ncol=2)

fig.suptitle("FDR for shifted vs shrinked data across datasets", y=0.98, fontsize=20)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("figures/fdr_shift_vs_shrink.png")
plt.show()

In [ ]:
fdr_shift, fdr_shrink

In [ ]:
def pareto_front(df1, df2):
    results = {}

    for dataset in df1.index:
        vals1 = df1.loc[dataset]
        vals2 = df2.loc[dataset]

        methods = df1.columns
        pareto_methods = []

        for m in methods:
            dominated = False
            for other in methods:
                if other == m:
                    continue

                # other dominates m if it's >= in both and > in at least one
                if (
                    vals1[other] >= vals1[m] and
                    vals2[other] >= vals2[m] and
                    (vals1[other] > vals1[m] or vals2[other] > vals2[m])
                ):
                    dominated = True
                    break

            if not dominated:
                pareto_methods.append(m)

        # collect values
        results[dataset] = {
            m: (vals1[m], vals2[m]) for m in pareto_methods
        }

    return results

In [ ]:
for dataset, best_methods in pareto_front(fdr_shift.T, fdr_shrink.T).items():
    print(f"Dataset: {dataset}")
    for method, (val1, val2) in best_methods.items():
        print(f"  Method: {method}, FDR Shift: {val1:.4f}, FDR Shrink: {val2:.4f}")

In [ ]:
from collections import defaultdict


counts = defaultdict(int)

for dataset, best_methods in pareto_front(fdr_shift.T, fdr_shrink.T).items():
    for method in best_methods.keys():
        counts[method] += 1

counts

## Export to latex

In [ ]:
cohen_d_shift = get_result_df(all_results_shift, "cohen_d")
cohen_d_shrink = get_result_df(all_results_shrink, "cohen_d")
fdr_shift = get_result_df(all_results_shift, "FDR")
fdr_shrink = get_result_df(all_results_shrink, "FDR")
silhouette_score_shift = get_result_df(all_results_shift, "slihouette_score_max")
silhouette_score_shrink = get_result_df(all_results_shrink, "slihouette_score_max")

In [ ]:
def df_to_latex_resized(df):
    latex = df.to_latex(
        index=True,
        escape=True,
        float_format="%.3f",
    )
    latex = latex.replace(
        r"\begin{tabular}",
        r"\resizebox{\textheight}{!}{\begin{tabular}"
    )
    latex = latex.replace(
        r"\end{tabular}",
        r"\end{tabular}}"
    )
    return latex

In [ ]:
from textwrap import dedent
import pandas as pd

def underline_max(df):
    df_copy = df.copy()
    
    for col in df_copy.columns:
        max_val = df_copy[col].max()
        df_copy[col] = df_copy[col].apply(
            lambda x: f"\\underline{{{x:.4f}}}" if x == max_val else f"{x:.4f}"
        )
    
    return df_copy

def df_to_multi_latex(df, block=6):
    cols = list(df.columns)
    parts = []
    
    for i in range(0, len(cols), block):
        sub = df[cols[i:i+block]]
        
        sub = underline_max(sub)  # <-- apply underline here
        
        latex_table = sub.to_latex(escape=False)  # IMPORTANT
        
        wrapped = dedent(f"""
        \\begin{{center}}
        {latex_table}
        \\end{{center}}
        """)
        
        parts.append(wrapped)
    
    return "\n\n".join(parts)

In [ ]:
for df, measure, distortion in [
    (cohen_d_shift, "Cohen's d", "Shift"),
    (cohen_d_shrink, "Cohen's d", "Shrink"),
    (fdr_shift, "FDR", "Shift"),
    (fdr_shrink, "FDR", "Shrink"),
    (silhouette_score_shift, "Silhouette Score", "Shift"),  
    (silhouette_score_shrink, "Silhouette Score", "Shrink")
]:
    df_fixed = df.copy()
    df_fixed.index = df_fixed.index.map(
        lambda x: str(x).replace('_', r'\_')
    )

    safe_measure = measure.lower().replace(' ', '_').replace('_', r'\_')

    print(f"\\section*{{{measure} for {distortion} Distortion}}")
    print(df_to_multi_latex(df))
    print("\n\\newpage\n")  # <-- page break here

## Extra visualizations

In [ ]:
import matplotlib.pyplot as plt

method_name, counts = np.unique(fdr_shift.idxmax().to_numpy(), return_counts=True)
orders = np.argsort(counts)[::-1]
method_name = method_name[orders]
counts = counts[orders]

plt.figure(figsize=(10, 6))
plt.bar(method_name, counts)
plt.xlabel("Embedding Method")
plt.ylabel("Count of best performance across datasets")
plt.title("Distribution of Best Embedding Methods for shift invariance")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("figures/best_dataset_count_shift.png")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

method_name, counts = np.unique(fdr_shrink.idxmax().to_numpy(), return_counts=True)
orders = np.argsort(counts)[::-1]
method_name = method_name[orders]
counts = counts[orders]

plt.figure(figsize=(10, 6))
plt.bar(method_name, counts)
plt.xlabel("Embedding Method")
plt.ylabel("Count of best performance across datasets")
plt.title("Distribution of Best Embedding Methods for shrink invariance")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("figures/best_dataset_count_shrink.png")
plt.show()